In [9]:
# Model1: XGBoost model to predict Dissolved Reactive Phosphorus (DRP)

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import r2_score

import optuna  # pip install optuna
import xgboost as xgb  # pip install xgboost


In [10]:
# Load DRP training dataset (features + target already joined)

data_path = "../../New Datasets/Combined/Final Datasets/drp_training_complete.csv"
full = pd.read_csv(data_path)

print("DRP Training dataset shape:", full.shape)
print("Columns:", list(full.columns))
full.head()


DRP Training dataset shape: (9319, 24)
Columns: ['latitude', 'longitude', 'month_fitted', 'swir16', 'swir22', 'red', 'NDMI', 'MNDWI', 'pet', 'aet', 'def', 'q', 'ppt', 'soil', 'srad', 'tmax', 'tmin', 'vap', 'vpd', 'ws', 'pdsi', 'esa_lccs_class', 'esa_change_count', 'dissolved_reactive_phosphorus']


,latitude,longitude,month_fitted,swir16,swir22,red,NDMI,MNDWI,pet,aet,...,srad,tmax,tmin,vap,vpd,ws,pdsi,esa_lccs_class,esa_change_count,dissolved_reactive_phosphorus
0,-34.405833,19.600556,42.031880,13580.000000,10717.000000,9095.500000,0.166424,-0.184320,132.300003,11.900001,...,263.801483,23.480000,12.179999,1.372,0.79,3.48,-2.59,120.0,0.0,15.962394
1,-34.405833,19.600556,38.341460,14055.575913,11746.892550,10363.898049,0.012617,-0.165667,70.400002,67.599998,...,158.596588,17.779999,7.240000,1.004,0.53,3.36,-3.30,120.0,0.0,7.812940
2,-34.405833,19.600556,53.145121,14090.844997,11846.217608,10413.610383,0.003518,-0.172979,163.000000,18.000000,...,323.498383,26.469999,15.740000,1.703,0.93,2.07,-3.49,120.0,0.0,12.787102
3,-34.405833,19.600556,38.539387,11536.500000,9401.000000,8631.000000,0.129337,-0.139379,51.400002,43.700001,...,113.603378,17.420000,8.400000,1.106,0.45,3.50,-1.60,120.0,0.0,2.278678
4,-34.405833,19.600556,39.406902,13263.500000,10589.000000,9197.500000,0.139460,-0.176520,88.200005,58.600002,...,195.097321,19.730000,10.200000,1.231,0.55,3.64,-1.40,120.0,0.0,18.831269


In [11]:
# Build feature matrix X and target y for DRP

# Columns to exclude from features
exclude_cols = {
    "dissolved_reactive_phosphorus",    # target
    "latitude", "longitude",            # spatial identifiers
}

base_feature_cols = [c for c in full.columns if c not in exclude_cols]
X_full = full[base_feature_cols].copy()
y = full["dissolved_reactive_phosphorus"]

print("Initial number of features:", len(base_feature_cols))
print("Features:", base_feature_cols)

# 1) Remove multicollinearity: drop one of each highly correlated pair
corr_matrix = X_full.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

high_corr_threshold = 0.95
cols_to_drop_mc = [col for col in upper.columns if any(upper[col] > high_corr_threshold)]

X_mc = X_full.drop(columns=cols_to_drop_mc)
feature_cols_mc = list(X_mc.columns)

print(f"Dropped {len(cols_to_drop_mc)} highly correlated features (>|{high_corr_threshold}|)")
print("Remaining features after multicollinearity reduction:", len(feature_cols_mc))

# Expose reduced set for feature selection
X_reduced_mc = X_mc.copy()
feature_cols_reduced_mc = feature_cols_mc

print("Example remaining feature columns:", feature_cols_reduced_mc[:10])


Initial number of features: 21
Features: ['month_fitted', 'swir16', 'swir22', 'red', 'NDMI', 'MNDWI', 'pet', 'aet', 'def', 'q', 'ppt', 'soil', 'srad', 'tmax', 'tmin', 'vap', 'vpd', 'ws', 'pdsi', 'esa_lccs_class', 'esa_change_count']
Dropped 2 highly correlated features (>|0.95|)
Remaining features after multicollinearity reduction: 19
Example remaining feature columns: ['month_fitted', 'swir16', 'red', 'NDMI', 'MNDWI', 'pet', 'aet', 'def', 'q', 'ppt']


In [12]:
# Feature selection via XGBoost feature importance (on multicollinearity-reduced set)

fs_model = xgb.XGBRegressor(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)

fs_model.fit(X_reduced_mc, y)

importances = fs_model.feature_importances_
fi_fs = pd.DataFrame({"feature": feature_cols_reduced_mc, "importance": importances})
fi_fs = fi_fs.sort_values("importance", ascending=False).reset_index(drop=True)
fi_fs["cum_importance"] = fi_fs["importance"].cumsum()

# Keep features that explain up to 95% of total importance, but ensure at least 25 features
importance_cutoff = 0.90
min_features = 20
selected = fi_fs[fi_fs["cum_importance"] <= importance_cutoff]["feature"].tolist()
if len(selected) < min_features:
    selected = fi_fs.head(min_features)["feature"].tolist()

X = X_reduced_mc[selected].copy()
feature_cols = selected

print("Total features after multicollinearity reduction:", len(feature_cols_reduced_mc))
print("Selected features after importance-based selection:", len(feature_cols))
print("Top selected features:", feature_cols[:15])


Total features after multicollinearity reduction: 19
Selected features after importance-based selection: 19
Top selected features: ['esa_lccs_class', 'esa_change_count', 'soil', 'vpd', 'ppt', 'vap', 'pdsi', 'MNDWI', 'aet', 'tmax', 'ws', 'pet', 'tmin', 'def', 'q']


In [13]:
# Baseline XGBoost model (for quick R² and importances)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = xgb.XGBRegressor(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train, y_train)

y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

print(f"Baseline Train R²: {r2_score(y_train, y_train_pred):.3f}")
print(f"Baseline Test  R²: {r2_score(y_test, y_test_pred):.3f}")

importances_base = model.feature_importances_
fi_base = pd.DataFrame({"feature": feature_cols, "importance": importances_base})
fi_base.sort_values("importance", ascending=False).head(20)


Baseline Train R²: 0.849
Baseline Test  R²: 0.546


,feature,importance
0,esa_lccs_class,0.104601
1,esa_change_count,0.088124
2,soil,0.069501
15,swir16,0.060125
11,pet,0.057356
3,vpd,0.053081
7,MNDWI,0.048955
17,red,0.048902
12,tmin,0.046965
13,def,0.046341


In [ ]:
# Stratified K-Fold + Optuna hyperparameter tuning for DRP model

n_bins = 10
y_strat = pd.qcut(y, q=n_bins, labels=False, duplicates='drop')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def objective(trial: optuna.trial.Trial) -> float:
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 800),
        "max_depth": trial.suggest_int("max_depth", 3, 6),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 10.0),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 2.0),
        "random_state": 42,
        "n_jobs": -1,
    }

    model = xgb.XGBRegressor(**params)

    cv_scores = []
    for train_idx, valid_idx in skf.split(X, y_strat):
        X_tr, X_val = X.iloc[train_idx], X.iloc[valid_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[valid_idx]

        model.fit(X_tr, y_tr)
        y_val_pred = model.predict(X_val)
        cv_scores.append(r2_score(y_val, y_val_pred))

    return float(np.mean(cv_scores))

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=40, show_progress_bar=True)

print("Best CV R²:", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")


[I 2026-02-27 04:12:20,551] A new study created in memory with name: no-name-cc804c6d-a053-428a-9304-fee9657aba52
Best trial: 0. Best value: 0.529832:   2%|▎         | 1/40 [00:05<03:36,  5.55s/it]

[I 2026-02-27 04:12:26,101] Trial 0 finished with value: 0.5298322603597899 and parameters: {'n_estimators': 350, 'max_depth': 6, 'learning_rate': 0.11433121755899102, 'subsample': 0.5516552865442172, 'colsample_bytree': 0.6010145751806986, 'min_child_weight': 2.879630598482652, 'gamma': 4.8855177372498, 'reg_alpha': 0.10771238334338462, 'reg_lambda': 0.5854722414031328}. Best is trial 0 with value: 0.5298322603597899.


Best trial: 1. Best value: 0.53024:   5%|▌         | 2/40 [00:10<03:24,  5.38s/it] 

[I 2026-02-27 04:12:31,364] Trial 1 finished with value: 0.5302404131777083 and parameters: {'n_estimators': 419, 'max_depth': 5, 'learning_rate': 0.25580625196755463, 'subsample': 0.9997543648211659, 'colsample_bytree': 0.7154962801281397, 'min_child_weight': 9.89071773778146, 'gamma': 0.7610353827017385, 'reg_alpha': 0.5742843090174999, 'reg_lambda': 0.2577090012671195}. Best is trial 1 with value: 0.5302404131777083.


Best trial: 1. Best value: 0.53024:   8%|▊         | 3/40 [00:13<02:26,  3.97s/it]

...

[I 2026-02-27 04:15:01,434] Trial 29 finished with value: 0.5382427259854632 and parameters: {'n_estimators': 608, 'max_depth': 6, 'learning_rate': 0.13272535011289452, 'subsample': 0.7039280524416126, 'colsample_bytree': 0.9317093645302065, 'min_child_weight': 2.8821152214168064, 'gamma': 4.5969685055514145, 'reg_alpha': 0.08595204590676919, 'reg_lambda': 0.7254337161220226}. Best is trial 15 with value: 0.5656910466275321.


Best trial: 15. Best value: 0.565691:  78%|███████▊  | 31/40 [02:45<00:37,  4.14s/it]

[I 2026-02-27 04:15:05,867] Trial 30 finished with value: 0.5539200800432769 and parameters: {'n_estimators': 658, 'max_depth': 6, 'learning_rate': 0.10975334293511937, 'subsample': 0.8432050304188837, 'colsample_bytree': 0.8632857726705332, 'min_child_weight': 7.06092149824743, 'gamma': 3.7518416198091513, 'reg_alpha': 0.00027310398694277394, 'reg_lambda': 1.9914981417442006}. Best is trial 15 with value: 0.5656910466275321.


Best trial: 15. Best value: 0.565691:  80%|████████  | 32/40 [02:50<00:35,  4.44s/it]

[I 2026-02-27 04:15:11,029] Trial 31 finished with value: 0.5604213714188832 and parameters: {'n_estimators': 725, 'max_depth': 6, 'learning_rate': 0.04343292245124262, 'subsample': 0.8927384958773169, 'colsample_bytree': 0.8965320637957541, 'min_child_weight': 1.937195398688409, 'gamma': 1.9255974849381765, 'reg_alpha': 0.12076803638616507, 'reg_lambda': 1.4807383324289691}. Best is trial 15 with value: 0.5656910466275321.


In [ ]:
# Train final DRP model with best hyperparameters and report R² + feature importances

best_params = study.best_params.copy()
best_params.update({"random_state": 42, "n_jobs": -1})

final_model = xgb.XGBRegressor(**best_params)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

final_model.fit(X_train, y_train)

y_train_pred = final_model.predict(X_train)
y_test_pred = final_model.predict(X_test)

r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)

print(f"Final DRP model Train R²: {r2_train:.3f}")
print(f"Final DRP model Test  R²: {r2_test:.3f}")

importances = final_model.feature_importances_
fi_tuned = pd.DataFrame({"feature": feature_cols, "importance": importances})
fi_tuned = fi_tuned.sort_values("importance", ascending=False)

print("\nTop 20 most important features for predicting DRP (tuned model):")
print(fi_tuned.head(20).to_string(index=False))

fi_tuned.head(20)


Final DRP model Train R²: 0.924
Final DRP model Test  R²: 0.559

Top 20 most important features for predicting DRP (tuned model):
         feature  importance
esa_change_count    0.150877
  esa_lccs_class    0.133836
            soil    0.077050
             vpd    0.054722
          swir16    0.051673
           MNDWI    0.045696
             pet    0.045127
             def    0.040001
            pdsi    0.039621
             vap    0.038198
             red    0.038086
             ppt    0.038063
              ws    0.038001
            tmin    0.037903
            tmax    0.036635
               q    0.036632
            NDMI    0.034937
             aet    0.034758
    month_fitted    0.028183


,feature,importance
1,esa_change_count,0.150877
0,esa_lccs_class,0.133836
2,soil,0.077050
3,vpd,0.054722
15,swir16,0.051673
7,MNDWI,0.045696
11,pet,0.045127
13,def,0.040001
6,pdsi,0.039621
5,vap,0.038198


In [ ]:
# Create predictions for DRP on validation data and update submission file

# Load current submission file (should have EC predictions already)
submission_path = "../../submission1.csv"
submission = pd.read_csv(submission_path)
print("Current submission shape:", submission.shape)
print("Submission columns:", list(submission.columns))

# VALIDATION CHECK: Ensure submission has exactly 200 rows
if submission.shape[0] != 200:
    print(f"\n⚠️  WARNING: Submission has {submission.shape[0]} rows instead of 200!")
    print("Loading fresh template instead...")
    submission = pd.read_csv("../../submission_template.csv")
    print(f"Loaded template with shape: {submission.shape}")

# Load validation features
val_features = pd.read_csv("../../New Datasets/Combined/Final Datasets/drp_validation.csv")
print("\nValidation dataset shape:", val_features.shape)
print("Validation columns:", list(val_features.columns))

# Build X_val using same DRP feature set
X_val = val_features[feature_cols].copy()
print("\nValidation features shape:", X_val.shape)
print(f"Using {len(feature_cols)} features: {feature_cols[:5]}...")

# Predict DRP for validation rows
drp_pred = final_model.predict(X_val)
print(f"\nGenerated {len(drp_pred)} DRP predictions")
print(f"DRP predictions - Min: {drp_pred.min():.2f}, Max: {drp_pred.max():.2f}, Mean: {drp_pred.mean():.2f}")

# CRITICAL: Match predictions to template by LAT/LON/DATE (not row order!)
val_features['DRP_prediction'] = drp_pred

# Standardize column names for merge
val_features_std = val_features.rename(columns={
    'latitude': 'Latitude',
    'longitude': 'Longitude',
    'sample_date': 'Sample Date'
})

# Merge predictions with submission by coordinates AND date
submission_with_drp = submission.merge(
    val_features_std[['Latitude', 'Longitude', 'Sample Date', 'DRP_prediction']],
    on=['Latitude', 'Longitude', 'Sample Date'],
    how='left'
)

# Update DRP column
submission_with_drp['Dissolved Reactive Phosphorus'] = submission_with_drp['DRP_prediction']
submission_with_drp = submission_with_drp.drop(columns=['DRP_prediction'])

# Ensure column order matches original
submission = submission_with_drp[submission.columns]

# VALIDATION CHECK: Ensure no missing predictions
missing_count = submission['Dissolved Reactive Phosphorus'].isnull().sum()
if missing_count > 0:
    print(f"\n⚠️  WARNING: {missing_count} locations without DRP predictions!")
    print("Missing locations:")
    print(submission[submission['Dissolved Reactive Phosphorus'].isnull()][['Latitude', 'Longitude', 'Sample Date']].head(10))
else:
    print("\n✓ All 200 locations have DRP predictions")

# Save updated submission file
submission.to_csv(submission_path, index=False)
print(f"\nUpdated submission file: {submission_path}")
print("Submission shape:", submission.shape)
print("\nFirst few rows of submission:")
submission.head()


Current submission shape: (200, 6)
Submission columns: ['Longitude', 'Latitude', 'Sample Date', 'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']

Validation dataset shape: (200, 23)
Validation columns: ['latitude', 'longitude', 'sample_date', 'swir16', 'swir22', 'red', 'NDMI', 'MNDWI', 'pet', 'aet', 'def', 'q', 'ppt', 'soil', 'srad', 'tmax', 'tmin', 'vap', 'vpd', 'ws', 'pdsi', 'esa_lccs_class', 'esa_change_count']


KeyError: "['month_fitted'] not in index"